# <font color="#2196F3">**PMC-Patients Clinical Data Collection**</font><br/>
### Retrieving Clinical Case Summaries from Hugging Face

This notebook demonstrates how to download and cache clinical patient summaries from the **[PMC-Patients Dataset](https://huggingface.co/datasets/zhengyun21/PMC-Patients)** (`https://huggingface.co/datasets/zhengyun21/PMC-Patients/resolve/main/PMC-Patients.csv`).

#### **Pipeline Overview**
- Source URL: `https://huggingface.co/datasets/zhengyun21/PMC-Patients/resolve/main/PMC-Patients.csv` (~545 MB full dataset).
- Efficient Ingestion: We download a 3.5 MB byte-range prefix (~1,500 patient case records) via HTTP `Range` requests.
- Output Cache: Saved locally to `eval/data/pmc_head.csv` for fast, reproducible downstream processing.

## 1️⃣ Import Necessary Libraries

In [1]:
import os
import sys
import csv
import json
import requests
import pandas as pd

# Allow large CSV text fields
csv.field_size_limit(10**7)

# Workspace path resolution
NOTEBOOK_DIR = os.getcwd()
DATA_PREP_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else NOTEBOOK_DIR
WORKSPACE_ROOT = os.path.abspath(os.path.join(DATA_PREP_DIR, "..", "..", ".."))

HF_URL = "https://huggingface.co/datasets/zhengyun21/PMC-Patients/resolve/main/PMC-Patients.csv"
CSV_CACHE_PATH = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "pmc_head.csv")
LOCAL_CSV_PATH = os.path.join(WORKSPACE_ROOT, "GTC25_DLI", "data", "pmc_head.csv")

print(f"Source URL:       {HF_URL}")
print(f"Backend CSV Path: {CSV_CACHE_PATH}")
print(f"Local CSV Path:   {LOCAL_CSV_PATH}")

Source URL:       https://huggingface.co/datasets/zhengyun21/PMC-Patients/resolve/main/PMC-Patients.csv
Backend CSV Path: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_head.csv
Local CSV Path:   /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/pmc_head.csv


## 2️⃣ Download and Cache PMC-Patients CSV Prefix
We fetch a byte-range prefix using an HTTP Range header to avoid waiting for the full 545 MB file.

In [2]:
def download_pmc_patients_prefix(url=HF_URL, out_paths=None, byte_count=3_500_000, force=False):
    """Download byte-range prefix from Hugging Face if not already present."""
    if out_paths is None:
        out_paths = [CSV_CACHE_PATH, LOCAL_CSV_PATH]
        
    primary_out = out_paths[0]
    os.makedirs(os.path.dirname(primary_out), exist_ok=True)

    if os.path.exists(primary_out) and os.path.getsize(primary_out) > 0 and not force:
        print(f"✅ Using existing cached CSV: {primary_out} ({os.path.getsize(primary_out):,} bytes)")
    else:
        print(f"Downloading {byte_count:,} bytes from {url}...")
        headers = {"Range": f"bytes=0-{byte_count - 1}"}
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        with open(primary_out, "wb") as f:
            f.write(resp.content)
        print(f"✅ Saved prefix ({len(resp.content):,} bytes) -> {primary_out}")

    # Mirror to other output paths if specified
    for p in out_paths[1:]:
        os.makedirs(os.path.dirname(p), exist_ok=True)
        if not os.path.exists(p) or force:
            with open(primary_out, "rb") as src_f, open(p, "wb") as dst_f:
                dst_f.write(src_f.read())
            print(f"✅ Mirrored CSV -> {p}")

    return primary_out

cached_file = download_pmc_patients_prefix()

✅ Using existing cached CSV: /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_head.csv (3,500,000 bytes)


## 3️⃣ Inspect CSV Schema and Record Samples

In [3]:
# Verify header columns
with open(cached_file, 'r', encoding='utf-8', errors='replace', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    sample_row = next(reader)

print("--- Header Columns ---")
for i, col in enumerate(header):
    print(f"  {i}: {col}")

print(f"\n--- Sample Row (Columns: {len(sample_row)}) ---")
for col, val in zip(header, sample_row):
    val_disp = val[:120] + "..." if len(val) > 120 else val
    print(f"  {col}: {val_disp}")

--- Header Columns ---
  0: patient_id
  1: patient_uid
  2: PMID
  3: file_path
  4: title
  5: patient
  6: age
  7: gender
  8: relevant_articles
  9: similar_patients

--- Sample Row (Columns: 10) ---
  patient_id: 0
  patient_uid: 7665777-1
  PMID: 33492400
  file_path: comm/PMC007xxxxxx/PMC7665777.xml
  title: Early Physical Therapist Interventions for Patients With COVID-19 in the Acute Care Hospital: A Case Report Series
  patient: This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea...
  age: [[60.0, 'year']]
  gender: M
  relevant_articles: {'32320506': 1, '32293716': 1, '23219649': 1, '30339549': 1, '17470624': 1, '32280973': 1, '34789437': 1, '30427933': 1,...
  similar_patients: {'7665777-2': 2, '7665777-3': 2, '7665777-4': 2, '7665777-5': 2, '7665777-6': 2, '7665777-7': 2, '7665777-8': 2, '766577...


## 4️⃣ Preview Dataset with Pandas

In [4]:
# Load preview into DataFrame
df_preview = pd.read_csv(cached_file, nrows=10, on_bad_lines='skip')
print(f"Loaded {len(df_preview)} sample rows.")
df_preview[['patient_id', 'age', 'gender', 'title']].head()

Loaded 10 sample rows.


,patient_id,age,gender,title
0,0,"[[60.0, 'year']]",M,Early Physical Therapist Interventions for Pat...
1,1,"[[39.0, 'year']]",M,Early Physical Therapist Interventions for Pat...
2,2,"[[57.0, 'year']]",M,Early Physical Therapist Interventions for Pat...
3,3,"[[69.0, 'year']]",M,Early Physical Therapist Interventions for Pat...
4,4,"[[57.0, 'year']]",M,Early Physical Therapist Interventions for Pat...
